# Residue-level scores for THOR variants

A scratch notebook for looking at *why* a variant scored the way it did: for every
position, the full distribution each scoring model puts over the 20 amino acids,
and how much probability the residue actually present got.

Feed it the zip that `generate_thor_colab.ipynb` downloads, or any THOR output
directory. Three scores may be in there, each on its own figure:

| Score | File | Varies per variant? |
|---|---|---|
| **ESM** | `*_scoring_probs.jsonl` (`--store_scoring_probs`) | yes, re-scored for each sequence |
| **ProteinMPNN** | `*_scoring_probs.jsonl` (needs `--structure`) | no, it describes the backbone |
| **Generator** | `*_variants_softmax.jsonl` (`--store_softmax`) | no, one per input sequence |

If a file is missing the notebook says so and skips that figure.

Both ESM and ProteinMPNN are used twice in a THOR run — to build the generator's
input features and to score what came out — so the probability file names its key
`Scoring_Model`: it only ever holds the scoring side. What they contributed as
features is in `*_run_metadata.json`, printed below when the bundle carries it.
Older bundles whose key is `model` are read unchanged.

## 0) Setup

In [ ]:
#@title Imports
import json, re, zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

try:
    import ipywidgets as W
    from IPython.display import display
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets not available - use the manual fallback at the bottom.")

AA_ORDER = "ACDEFGHIKLMNPQRSTVWY"   # the column order THOR writes
print("ready")

## 1) Load a bundle

In [ ]:
#@title Point at a zip (or a directory) and unpack it
#@markdown A `*_thor_outputs.zip` from the Colab notebook, a zip you made yourself,
#@markdown or a directory that already holds the `*_variants.fasta` and `*.jsonl` files.
BUNDLE = "" #@param {type:"string"}
WORK_DIR = "/tmp/thor_inspect" #@param {type:"string"}

work = Path(WORK_DIR)
work.mkdir(parents=True, exist_ok=True)

if not BUNDLE:
    # Nothing given: offer an upload in Colab, otherwise look in the output dir.
    try:
        from google.colab import files
        print("Choose a THOR output zip...")
        up = files.upload()
        BUNDLE = str(next(iter(up)))
    except ImportError:
        raise ValueError("Set BUNDLE to a zip or a directory.")

src = Path(BUNDLE)
if not src.exists():
    raise FileNotFoundError(f"No such file or directory: {src}")

if src.is_dir():
    DATA_DIR = src
else:
    with zipfile.ZipFile(src) as z:
        z.extractall(work)
    DATA_DIR = work

print(f"Reading from {DATA_DIR}")
for p in sorted(DATA_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(DATA_DIR)}  ({p.stat().st_size/1024:.1f} KB)")

In [ ]:
#@title Parse the sequences and the probability files

def parse_fasta(path):
    """The variant FASTA, with the header metadata kept as columns."""
    recs, label, meta, seq = [], None, {}, []

    def flush():
        if label is not None:
            recs.append({"label": label, "seq": "".join(seq), **meta})

    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            flush()
            parts = [p.strip() for p in line[1:].split("|")]
            label, seq, meta = parts[0], [], {}
            for part in parts[1:]:
                if "=" in part:
                    k, v = part.split("=", 1)
                    meta[k.strip()] = v.strip()
        else:
            seq.append(line)
    flush()
    return recs


fastas = sorted(DATA_DIR.rglob("*_variants.fasta"))
if not fastas:
    raise FileNotFoundError(f"No *_variants.fasta under {DATA_DIR}")
if len(fastas) > 1:
    print(f"[info] {len(fastas)} FASTAs found; using {fastas[0].name}. "
          f"Change FASTA_INDEX below to pick another.")
FASTA_INDEX = 0
fasta_path = fastas[FASTA_INDEX]

RECORDS = parse_fasta(fasta_path)
BY_LABEL = {r["label"]: r for r in RECORDS}
WT = next((r for r in RECORDS if r.get("type") == "wt"), RECORDS[0])
WT_SEQ = WT["seq"]

# ESM is written per emitted sequence; ProteinMPNN once for the backbone.
# The key is "Scoring_Model" - these models also act as feature models, and this
# file is only ever the scoring side. Bundles written before that rename say
# "model" instead, so both are read.
ESM, MPNN, GEN = {}, None, {}
for f in DATA_DIR.rglob("*_scoring_probs.jsonl"):
    for line in f.read_text().splitlines():
        if not line.strip():
            continue
        d = json.loads(line)
        probs = np.asarray(d["probs"], dtype=np.float32)
        which = d.get("Scoring_Model", d.get("model"))
        if which == "esm":
            ESM[d["label"]] = probs
        elif which == "proteinmpnn":
            MPNN = probs

# The generator softmax has 21 columns; the last one is not an amino acid.
for f in DATA_DIR.rglob("*_variants_softmax.jsonl"):
    for line in f.read_text().splitlines():
        d = json.loads(line)
        GEN[d["id"].rsplit("_rep", 1)[0]] = np.asarray(d["probs"], dtype=np.float32)[:, :20]

print(f"{len(RECORDS)} sequences from {fasta_path.name}  (wildtype: {WT['label']}, {len(WT_SEQ)} aa)")
print(f"ESM distributions     : {len(ESM)}" if ESM else
      "ESM distributions     : none  (re-run generate_thor.py with --store_scoring_probs)")
print(f"ProteinMPNN           : {'yes, ' + str(len(MPNN)) + ' positions' if MPNN is not None else 'none  (needs --structure)'}")
print(f"Generator softmax     : {len(GEN)}" if GEN else
      "Generator softmax     : none  (re-run with --store_softmax)")

# Written beside the outputs since the metadata commit; absent from older bundles.
META, STRUCTURE_IN_BUNDLE = {}, None
meta_files = sorted(DATA_DIR.rglob("*_run_metadata.json"))
if meta_files:
    META = json.loads(meta_files[0].read_text())
    feat = META.get("feature_models", {})
    print(f"\nRun settings ({meta_files[0].name})")
    print(f"  generator   : {feat.get('generator', {}).get('run_dir', '?')}"
          f"  temp {feat.get('generator', {}).get('temperature', '?')}")
    print(f"  features    : ESM {feat.get('esm', {}).get('weight_in_blend', 1.0):.2f}"
          f" / ProteinMPNN {feat.get('proteinmpnn', {}).get('weight_in_blend', 0.0):.2f}")
    if META.get("structure"):
        st = META["structure"]
        print(f"  structure   : {st.get('file')} chain {st.get('chain')}"
              f"  ({st.get('residues')} residues)")
for cand in DATA_DIR.rglob("*_structure.*"):
    STRUCTURE_IN_BUNDLE = cand
    print(f"  bundled file: {cand.name}")
    break


def models_for(label):
    """(name, probs) for every score available for this variant."""
    out = []
    if label in ESM:
        out.append(("ESM", ESM[label]))
    for key, probs in GEN.items():
        if label.startswith(key):
            out.append(("Generator", probs))
            break
    if MPNN is not None:
        out.append(("ProteinMPNN", MPNN))
    return out


def substitutions(label):
    """Positions (0-based) where this variant differs from the wildtype."""
    return [i for i, (a, b) in enumerate(zip(BY_LABEL[label]["seq"], WT_SEQ)) if a != b]

## 2) How the rows are ordered

Alphabetical order scatters chemically similar residues all over the y axis. Here
they are grouped by side-chain chemistry instead, so neighbouring rows are
interchangeable substitutions and a solid block of signal reads as *"this position
wants a hydrophobic"* rather than as five unrelated letters:

| Group | Residues |
|---|---|
| hydrophobic | A V L I M |
| aromatic | F W Y |
| polar | S T N Q |
| acidic | D E |
| basic | K R H |
| special | C G P |

Histidine sits with the basic group, which is the usual convention, though it is
only partly charged at neutral pH. Cysteine, glycine and proline are grouped as
"special" because what they do to a structure has little to do with polarity.

In [ ]:
#@title Grouping, colours and the plotting function

AA_GROUPS = [
    ("hydrophobic", "AVLIM"),
    ("aromatic", "FWY"),
    ("polar", "STNQ"),
    ("acidic", "DE"),
    ("basic", "KRH"),
    ("special", "CGP"),
]
GROUPED_AA = "".join(aa for _, aa in AA_GROUPS)
ROW_OF = {aa: i for i, aa in enumerate(GROUPED_AA)}
REORDER = [AA_ORDER.index(aa) for aa in GROUPED_AA]
assert sorted(GROUPED_AA) == sorted(AA_ORDER), "the grouping must cover all 20 residues"

SURFACE, INK, INK_SOFT = "#fcfcfb", "#0b0b0b", "#52514e"
GRID, RULE, WARMER, MUTED = "#e3e2de", "#c9c7c2", "#eb6834", "#9a9894"
MODEL_HUE = {"ESM": "#2a78d6", "Generator": "#1baf7a", "ProteinMPNN": "#6250d6"}

# Colour and height limits come from the whole sequence, not the window, so that
# scrolling does not silently rescale the picture under you.
_SCALE = {}


def scale_for(name, probs):
    if name not in _SCALE:
        _SCALE[name] = (float(np.quantile(probs, 0.995)), float(probs.max()))
    return _SCALE[name]


def plot_window(probs, var_seq, cols, model_name, variant_label, contiguous=True):
    """One figure for one score. `cols` are the 0-based positions to draw."""
    cols = list(cols)
    n = len(cols)
    var_win = "".join(var_seq[i] for i in cols)
    wt_win = "".join(WT_SEQ[i] for i in cols)
    win = probs[cols]
    grid = win[:, REORDER].T
    hue = MODEL_HUE.get(model_name, "#2a78d6")
    vmax, track_max = scale_for(model_name, probs)
    cmap = LinearSegmentedColormap.from_list("ramp", [SURFACE, hue])

    p_var = win[np.arange(n), [AA_ORDER.index(a) for a in var_win]]
    p_wt = win[np.arange(n), [AA_ORDER.index(a) for a in wt_win]]
    is_sub = np.array([a != b for a, b in zip(var_win, wt_win)])

    # Scattered columns get wider cells, since there are far fewer of them.
    fig_w = max(9.5, n * 0.215) if contiguous else max(8.0, n * 0.55)
    fig = plt.figure(figsize=(fig_w, 7.6), facecolor=SURFACE)
    # Margins are fixed in inches, so the group labels and the colourbar keep
    # their room whatever the figure width works out to.
    gs = fig.add_gridspec(2, 1, height_ratios=[1, 2.7], hspace=0.07,
                          left=1.57 / fig_w, right=1 - 0.95 / fig_w,
                          top=0.855, bottom=0.085)
    ax_t = fig.add_subplot(gs[0])
    ax_h = fig.add_subplot(gs[1], sharex=ax_t)

    # --- how likely is the residue that is actually there -------------------
    ax_t.set_facecolor(SURFACE)
    if contiguous:
        for i in np.flatnonzero(is_sub):
            ax_t.axvspan(i - 0.5, i + 0.5, color=WARMER, alpha=0.09, lw=0, zorder=0)
    ax_t.bar(range(n), p_var, width=0.72,
             color=[WARMER if s else MUTED for s in is_sub], zorder=3)
    if is_sub.any():
        sub_i = np.flatnonzero(is_sub)
        ax_t.scatter(sub_i, p_wt[sub_i], s=34, marker="_", color=INK,
                     linewidths=1.7, zorder=5)
        for i in sub_i:
            ax_t.annotate(f"{wt_win[i]}{cols[i] + 1}{var_win[i]}",
                          (i, max(p_var[i], p_wt[i])), textcoords="offset points",
                          xytext=(0, 6), ha="center", fontsize=7.5, color=WARMER,
                          rotation=90, zorder=6)
    ax_t.set_ylim(0, track_max * 1.28)
    ax_t.set_ylabel("probability of\nthe residue present", fontsize=8.5, color=INK_SOFT)
    ax_t.yaxis.grid(True, color=GRID, lw=0.8)
    ax_t.set_axisbelow(True)
    for s in ("top", "right", "left"):
        ax_t.spines[s].set_visible(False)
    ax_t.spines["bottom"].set_color(GRID)
    ax_t.tick_params(colors=INK_SOFT, labelsize=8, length=0, labelbottom=False)

    # --- the full distribution over all 20 residues -------------------------
    im = ax_h.imshow(grid, aspect="auto", cmap=cmap, vmin=0, vmax=vmax,
                     interpolation="nearest", extent=(-0.5, n - 0.5, 19.5, -0.5))
    ax_h.set_yticks(range(20))
    ax_h.set_yticklabels(list(GROUPED_AA), fontsize=8.5, color=INK, family="monospace")

    row = 0
    for name, members in AA_GROUPS:
        if row:
            ax_h.axhline(row - 0.5, color=RULE, lw=1.2, zorder=5)
        ax_h.text(-0.085, row + len(members) / 2 - 0.5, name, fontsize=8, color=INK_SOFT,
                  va="center", ha="right", transform=ax_h.get_yaxis_transform())
        row += len(members)

    for i, (a, b) in enumerate(zip(var_win, wt_win)):
        on_dark = grid[ROW_OF[a], i] > 0.55 * vmax
        ax_h.add_patch(Rectangle(
            (i - 0.5, ROW_OF[a] - 0.5), 1, 1, fill=False,
            ec=WARMER if a != b else (SURFACE if on_dark else INK),
            lw=1.8 if a != b else 1.0, zorder=6))
        if a != b:  # where the substitution came from
            ax_h.add_patch(Rectangle((i - 0.5, ROW_OF[b] - 0.5), 1, 1, fill=False,
                                     ec=INK_SOFT, lw=1.1, ls=(0, (1.4, 1.2)), zorder=6))

    ax_h.set_xlim(-0.5, n - 0.5)
    ax_h.set_xticks(range(n))
    ax_h.set_xticklabels(list(var_win), fontsize=7.5, family="monospace")
    for i, tick in enumerate(ax_h.get_xticklabels()):
        tick.set_color(WARMER if is_sub[i] else INK)
    if contiguous:
        step = 10 if n > 40 else 5
        label_at = [i for i in range(n) if (cols[i] + 1) % step == 0]
    else:
        label_at = range(n)  # the columns are not adjacent, so number every one
    for i in label_at:
        ax_h.annotate(str(cols[i] + 1), (i, -0.055),
                      xycoords=ax_h.get_xaxis_transform(), ha="center", va="top",
                      fontsize=7, color=WARMER if is_sub[i] else INK_SOFT,
                      rotation=0 if contiguous else 90)
    ax_h.tick_params(colors=INK_SOFT, length=0, pad=2)
    for s in ax_h.spines.values():
        s.set_visible(False)

    cax = fig.add_axes([1 - 0.78 / fig_w, 0.085, 0.115 / fig_w, 0.54])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(f"{model_name} probability", fontsize=8, color=INK_SOFT)
    cb.outline.set_visible(False)
    cb.ax.tick_params(colors=INK_SOFT, labelsize=7.5, length=0)

    handles = [
        Rectangle((0, 0), 1, 1, fill=False, ec=INK, lw=1.0),
        Rectangle((0, 0), 1, 1, fill=False, ec=INK_SOFT, lw=1.1, ls=(0, (1.4, 1.2))),
        Rectangle((0, 0), 1, 1, fill=False, ec=WARMER, lw=1.8),
        Line2D([0], [0], marker="_", color=INK, lw=0, markersize=9, markeredgewidth=1.7),
    ]
    fig.legend(handles, ["residue in this variant", "wildtype residue", "substitution",
                         "p(wildtype) at a substitution"],
               loc="upper right", bbox_to_anchor=(1 - 0.45 / fig_w, 0.945),
               ncol=4 if fig_w > 9 else 2, frameon=False,
               fontsize=8, labelcolor=INK_SOFT, handletextpad=0.5, columnspacing=1.5)

    n_sub = int(is_sub.sum())
    where = (f"residues {cols[0] + 1}-{cols[-1] + 1}" if contiguous
             else f"{n} substitution{'s' if n != 1 else ''} only")
    fig.suptitle(
        f"{variant_label}   ·   {model_name}   ·   {where}"
        + (f"   ({n_sub} substitution{'s' if n_sub != 1 else ''} in view)" if contiguous else ""),
        fontsize=11.5, color=INK, x=0.012, ha="left", y=0.985)
    return fig


def show(variant_label, start=0, width=50, models=None, subs_only=False):
    """One figure per score. `subs_only` drops every position that did not change."""
    var_seq = BY_LABEL[variant_label]["seq"]
    available = models_for(variant_label)
    if models:
        available = [(n, p) for n, p in available if n in models]
    if not available:
        print("No probability files for this variant - see the load cell above.")
        return

    if subs_only:
        cols = substitutions(variant_label)
        if not cols:
            print(f"{variant_label} is identical to the wildtype - nothing to show.")
            return
    else:
        cols = range(start, min(start + width, len(var_seq)))

    for name, probs in available:
        if len(probs) < len(var_seq):
            print(f"[warn] {name} covers {len(probs)} of {len(var_seq)} positions; skipping")
            continue
        plot_window(probs, var_seq, cols, name, variant_label, contiguous=not subs_only)
        plt.show()


print("plotting ready")

## 2b) Missing ProteinMPNN probabilities? Add them here

`generate_thor.py` only wrote the ProteinMPNN line when it had actually computed
the backbone distribution, which before the fix meant `--mpnn_weight 0` runs
produced an MPNN *score* per variant but no probabilities. If the cell above
reported `ProteinMPNN: none` and you still have the structure, this recomputes
them and writes them into the bundle's `*_scoring_probs.jsonl` in the same
format `generate_thor.py` uses, so the rest of the notebook picks them up.

Recent bundles carry the structure they were run against, and the cell uses it
without being told. Fill `STRUCTURE` in only for an older bundle.

They depend only on the backbone, so recomputing later gives the same numbers as
the original run would have — provided the same chain and the same model.

In [ ]:
#@title (Optional) Compute ProteinMPNN probabilities from a structure
#@markdown Leave `STRUCTURE` empty to use the structure bundled with the run, if
#@markdown there is one; this cell is skipped when there is neither.
STRUCTURE = "" #@param {type:"string"}
MPNN_CHAIN = "A" #@param {type:"string"}
MPNN_MODEL = "v_48_020" #@param ["v_48_020", "v_48_010", "v_48_030"]
MPNN_SOLUBLE = False #@param {type:"boolean"}
THOR_REPO = "/tmp/THOR-repo" #@param {type:"string"}
MPNN_REPO = "/tmp/ProteinMPNN" #@param {type:"string"}

struct_path = Path(STRUCTURE) if STRUCTURE else STRUCTURE_IN_BUNDLE

if struct_path is None:
    print("No structure given and none in the bundle - skipping.")
elif MPNN is not None:
    print("ProteinMPNN probabilities are already loaded. Clear MPNN first to recompute.")
else:
    import subprocess, sys

    # Reuse the clones if this runtime already has them, otherwise fetch.
    if not Path(THOR_REPO).exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Umbriona/THOR", THOR_REPO], check=True)
    if not (Path(MPNN_REPO) / "protein_mpnn_run.py").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/dauparas/ProteinMPNN", MPNN_REPO], check=True)
    sys.path.insert(0, str(Path(THOR_REPO) / "src" / "scripts"))

    from data_processing.compute_mpnn_features import compute_structure_probs, list_chains

    struct = Path(struct_path)
    if not struct.exists():
        raise FileNotFoundError(f"No such structure: {struct}")
    chains = list_chains(str(struct))
    chain = MPNN_CHAIN.strip() or chains[0]
    if chain not in chains:
        raise ValueError(f"Chain {chain!r} not in {struct.name}; found {chains}")
    if not MPNN_CHAIN.strip() and len(chains) > 1:
        print(f"[warn] {struct.name} has chains {chains}; using {chain}")

    print(f"Running ProteinMPNN on {struct.name} chain {chain}...")
    probs, struct_seq = compute_structure_probs(
        str(struct), chain, MPNN_REPO,
        model_name=MPNN_MODEL, use_soluble_model=MPNN_SOLUBLE)

    # Positions must line up, or the heatmap would label the wrong residues.
    if len(struct_seq) != len(WT_SEQ):
        raise ValueError(
            f"Chain {chain} has {len(struct_seq)} resolved residues but the wildtype is "
            f"{len(WT_SEQ)} aa. Unresolved residues shift every position after the gap.")
    if struct_seq != WT_SEQ:
        diff = next(i for i, (a, b) in enumerate(zip(struct_seq, WT_SEQ)) if a != b)
        raise ValueError(
            f"Chain {chain} disagrees with the wildtype at position {diff + 1}: "
            f"{struct_seq[diff]} vs {WT_SEQ[diff]}. Wrong structure for this run?")

    MPNN = probs
    _SCALE.pop("ProteinMPNN", None)  # limits were cached from the old data

    # Same schema generate_thor.py writes, merged into the existing file.
    target = next(iter(DATA_DIR.rglob("*_scoring_probs.jsonl")), None)
    if target is None:
        target = DATA_DIR / f"{fasta_path.name.split('_variants')[0]}_scoring_probs.jsonl"
        kept = []
    else:
        kept = [l for l in target.read_text().splitlines() if l.strip() and
                json.loads(l).get("Scoring_Model", json.loads(l).get("model")) != "proteinmpnn"]
    line = json.dumps({
        "Scoring_Model": "proteinmpnn",
        "label": f"structure_chain_{chain}",
        "aa_order": AA_ORDER,
        "probs": [[round(float(v), 6) for v in row] for row in probs],
    })
    target.write_text("\n".join([line] + kept) + "\n")
    print(f"{len(probs)} positions written to {target}")

## 3) Browse

Pick a variant, then drag the slider to scroll along the sequence. **Jump to**
moves the window straight to a substitution, which beats hunting for eight
changes in a few hundred positions.

**substitutions only** drops every position that did not change, which turns a
few hundred columns into however many substitutions the variant has. The columns
are then no longer adjacent, so each one is numbered.

Reading a panel: a column is one position, and the colour is how much probability
that model gave each residue there. The boxed cell is what the variant has; a
dotted box is what the wildtype had. The track on top pulls out the boxed value —
**a short orange bar under a high black dash is a substitution the model dislikes**,
exactly the case worth a second look.

In [ ]:
#@title Interactive viewer
VARIANTS = [r["label"] for r in RECORDS]
DEFAULT = next((v for v in VARIANTS if BY_LABEL[v].get("type") != "wt"), VARIANTS[0])

if HAVE_WIDGETS:
    w_variant = W.Dropdown(options=VARIANTS, value=DEFAULT, description="Variant:",
                           layout=W.Layout(width="420px"))
    w_width = W.Dropdown(options=[30, 40, 50, 60, 80, 100], value=50,
                         description="Window:", layout=W.Layout(width="200px"))
    w_start = W.IntSlider(min=0, max=max(0, len(WT_SEQ) - 50), step=5, value=0,
                          description="Start:", continuous_update=False,
                          layout=W.Layout(width="620px"))
    w_models = W.SelectMultiple(options=[n for n, _ in models_for(DEFAULT)] or ["(none)"],
                                value=tuple(n for n, _ in models_for(DEFAULT)),
                                description="Scores:", rows=3,
                                layout=W.Layout(width="260px"))
    w_jump = W.Dropdown(options=[("(jump to a substitution)", -1)], value=-1,
                        description="Jump to:", layout=W.Layout(width="420px"))
    w_prev, w_next = W.Button(description="◀", layout=W.Layout(width="45px")), \
                     W.Button(description="▶", layout=W.Layout(width="45px"))
    w_subs = W.Checkbox(value=False, description="substitutions only", indent=False,
                        layout=W.Layout(width="200px"))

    def refresh_variant(*_):
        """Window bounds and the jump list belong to the selected variant."""
        label = w_variant.value
        w_start.max = max(0, len(BY_LABEL[label]["seq"]) - w_width.value)
        w_start.step = max(1, w_width.value // 5)
        names = [n for n, _ in models_for(label)]
        w_models.options = names or ["(none)"]
        w_models.value = tuple(names)
        var = BY_LABEL[label]["seq"]
        opts = [("(jump to a substitution)", -1)]
        opts += [(f"{WT_SEQ[i]}{i+1}{var[i]}", i) for i in substitutions(label)]
        w_jump.options = opts
        w_jump.value = -1
        set_scroll_enabled()

    def set_scroll_enabled(*_):
        """Scrolling means nothing when only the substituted columns are drawn."""
        off = w_subs.value
        for w in (w_start, w_width, w_jump, w_prev, w_next):
            w.disabled = off

    def do_jump(change):
        if change["new"] is not None and change["new"] >= 0:
            w_start.value = max(0, min(w_start.max, change["new"] - w_width.value // 2))

    def step(delta):
        return lambda _: setattr(w_start, "value",
                                 max(0, min(w_start.max, w_start.value + delta * w_width.value)))

    w_variant.observe(refresh_variant, names="value")
    w_width.observe(refresh_variant, names="value")
    w_subs.observe(set_scroll_enabled, names="value")
    w_jump.observe(do_jump, names="value")
    w_prev.on_click(step(-1))
    w_next.on_click(step(+1))
    refresh_variant()

    out = W.interactive_output(
        lambda variant, start, width, models, subs_only:
            show(variant, start, width, list(models), subs_only),
        {"variant": w_variant, "start": w_start, "width": w_width,
         "models": w_models, "subs_only": w_subs},
    )
    display(W.VBox([
        W.HBox([w_variant, w_width, w_subs]),
        W.HBox([w_jump, w_models]),
        W.HBox([w_prev, w_start, w_next]),
        out,
    ]))
else:
    print("ipywidgets missing - calling show() directly instead.")
    show(DEFAULT, subs_only=True)

## 4) Fallback and export

No widgets, or want a file to keep? Call `show()` yourself, or save every window
that contains a substitution.

In [ ]:
#@title Save the windows around each substitution
SAVE_VARIANT = "" #@param {type:"string"}
SAVE_WIDTH = 50 #@param {type:"integer"}
SAVE_DIR = "/tmp/thor_inspect/figures" #@param {type:"string"}

label = SAVE_VARIANT or DEFAULT
out_dir = Path(SAVE_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

var_seq = BY_LABEL[label]["seq"]
subs = substitutions(label)
if not subs:
    print(f"{label} is identical to the wildtype - nothing to save.")
else:
    # One window per substitution, skipping any already visible in the last one.
    starts, covered_to = [], 0
    for i in subs:
        if starts and i < covered_to:
            continue
        s = max(0, min(len(var_seq) - SAVE_WIDTH, i - SAVE_WIDTH // 2))
        starts.append(s)
        covered_to = s + SAVE_WIDTH
    print(f"{label}: {len(subs)} substitution(s) in {len(starts)} window(s)")
    for s in starts:
        for name, probs in models_for(label):
            if len(probs) < len(var_seq):
                continue
            end = min(s + SAVE_WIDTH, len(var_seq))
            fig = plot_window(probs, var_seq, range(s, end), name, label)
            path = out_dir / f"{label}_{name}_{s + 1}-{end}.png"
            fig.savefig(path, dpi=150, facecolor=SURFACE)
            plt.close(fig)
            print("  ", path.name)

    # And one figure holding every substitution at once.
    for name, probs in models_for(label):
        if len(probs) < len(var_seq):
            continue
        fig = plot_window(probs, var_seq, subs, name, label, contiguous=False)
        path = out_dir / f"{label}_{name}_substitutions.png"
        fig.savefig(path, dpi=150, facecolor=SURFACE)
        plt.close(fig)
        print("  ", path.name)

# show(label, subs_only=True)                           # substitutions only
# show(label, start=120, width=60)                      # one window, all scores
# show(label, start=120, width=60, models=["ESM"])      # one score only